# SR27_S3 – Laufvergleich und Dokumentation

In [0]:
import pypsa
import pandas as pd
from pathlib import Path


path_og = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/original"
path_mod_1 = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified_v1/"
path_mod_2 = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-2/"
path_mod_3 = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-3/"
path_mod_4 = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-4/"

def laden(path, year):
    return pypsa.Network(
        str(Path(path) / f"base_s_27__none_{year}.nc")
    )

def leistung(n, carrier, fest=False):
    l = n.links
    mask = (
        l.carrier.isin(carrier)
        & l.bus1.str.startswith("DE", na=False)
        & l.bus1.map(n.buses.carrier).eq("AC")
        & l.active
    )
    if fest:
        mask &= ~l.p_nom_extendable

    p = l.p_nom.where(~l.p_nom_extendable, l.p_nom_opt)
    return (p[mask] * l.loc[mask, "efficiency"]).sum() / 1e3

## Anpassung 1 prüfen

In [0]:
rows = []

for year in [2040, 2045]:
    n = laden(path_mod_2, year)
    l = n.links

    mask = (
        l.carrier.eq("H2 Electrolysis")
        & l.bus0.str.startswith("DE", na=False)
        & l.bus0.map(n.buses.carrier).eq("AC")
        & l.active
    )

    if not mask.any():
        raise ValueError(f"{year}: Keine passenden Elektrolyse-Links gefunden.")

    p = l.p_nom.where(~l.p_nom_extendable, l.p_nom_opt)
    gw = float(p.loc[mask].sum() / 1e3)

    rows.append({
        "Jahr": year,
        "Elektrolyse GW_el": gw,
        "Mindestziel GW_el": 17.0,
        "Mindestens 17 GW": "ja" if gw >= 17.0 - 0.001 else "nein",
    })

display(pd.DataFrame(rows).round(3))

Damit ist auch die Elektrolysevorgabe bestätigt: 2040 und 2045 werden jeweils 17 GW_el erreicht. Die neue H₂-Kraftwerksbedingung hat diese Vorgabe nicht beeinträchtigt.


## Anpassung 2 prüfen

In [0]:
gas = ["OCGT", "CCGT", "urban central gas CHP",
       "urban central gas CHP CC"]
h2 = ["H2 OCGT", "H2 CCGT", "urban central H2 CHP",
      "H2 retrofit OCGT", "H2 retrofit CCGT",
      "urban central H2 retrofit CHP"]

rows = []

for year in [2040, 2045]:
    for lauf, path in [("OG", path_og), ("Mod 1", path_mod_1), ("Mod 2", path_mod_2)]:
        n = laden(path, year)
        methan = leistung(n, gas, fest=True)

        rows.append({
            "Lauf": lauf,
            "Jahr": year,
            "Methan fest GW_el": methan,
            "Gas-KWK fest GW_el": leistung(
                n, ["urban central gas CHP", "urban central gas CHP CC"],
                fest=True
            ),
            "H2-Retrofit fest GW_el": leistung(
                n, ["H2 retrofit OCGT", "H2 retrofit CCGT"], fest=True
            ),
            "Methan + H2 fest GW_el": leistung(n, gas + h2, fest=True),
            "19 GW erreicht": (
                "ja" if abs(methan - 19) < 0.05 else "nein"
            ) if year == 2045 else "nicht gefordert",
                    })

display(pd.DataFrame(rows).round(3))

**Modified-2 funktioniert hinsichtlich der Teilumrüstung genau wie beabsichtigt:**
- 2045 sinkt der feste Methanbestand von 27,957 auf exakt 19,000 GW_el.
- Gleichzeitig entstehen 8,957 GW_el H₂-Retrofit.
- Methan + H₂ fest bleibt mit 28,414 GW_el unverändert. Die Leistung wurde also umgerüstet, nicht entfernt.
- Die Gas-KWK bleibt mit 15,410 GW_el unverändert. Auch diese Anforderung ist erfüllt.
- 2040 bleibt Mod 2 gegenüber Mod 1 unverändert. Das passt zu deinem Ergebnis.
- Damit können wir den Teil „fester Methanbestand / Retrofit“ als erfolgreich abhaken.

In [0]:
rows = []

for year in [2040, 2045]:
    for lauf, path in [("OG", path_og), ("Mod 1", path_mod_1), ("Mod 2", path_mod_2)]:
        n = laden(path, year)

        rows.append({
            "Lauf": lauf,
            "Jahr": year,
            "Methan gesamt GW_el": leistung(n, gas),
            "H2 gesamt GW_el": leistung(n, h2),
            "Thermisch gesamt GW_el": leistung(n, gas + h2),
        })

display(pd.DataFrame(rows).round(3))

**Modified-2 macht technisch das Richtige, erreicht aber die Gesamtzielwerte noch nicht:**
- 2040: praktisch keine Änderung, 30,2 GW Methan + 0,46 GW H₂ statt Ziel 50 + 10 GW.
- 2045: Retrofit funktioniert, aber die Optimierung baut zusätzlich rund 1,06 GW Methan zu. Deshalb landen wir bei 20,06 statt 19 GW Methan.
- H₂ steigt auf 9,41 GW, bleibt aber deutlich unter dem Ziel von 41 GW.
- Insgesamt kommen wir 2045 nur auf 29,47 statt 60 GW thermisch.

In [0]:
n = laden(path_mod_2, 2045)

l = n.links
mask = (
    l.carrier.isin(gas + h2)
    & l.bus1.str.startswith("DE", na=False)
    & l.bus1.map(n.buses.carrier).eq("AC")
    & l.active
)

p_el = l.p_nom.where(~l.p_nom_extendable, l.p_nom_opt) * l.efficiency

check = pd.DataFrame({
    "carrier": l.loc[mask, "carrier"],
    "extendable": l.loc[mask, "p_nom_extendable"],
    "GW_el": p_el.loc[mask] / 1e3,
})

display(
    check.groupby(["carrier", "extendable"])["GW_el"]
         .sum()
         .reset_index()
         .round(3)
)

## Anpassung 3 prüfen

In [0]:
rows = []

for year, ziel in [(2040, 10.0), (2045, 41.0)]:
    n = laden(path_mod_3, year)
    h2_gw = float(leistung(n, h2))
    methan_gw = float(leistung(n, gas))

    rows.append({
        "Jahr": year,
        "H2 gesamt GW_el": h2_gw,
        "H2 Ziel GW_el": ziel,
        "H2 Abweichung GW_el": h2_gw - ziel,
        "H2 Ziel erreicht": "ja" if abs(h2_gw - ziel) < 0.01 else "nein",
        "Methan gesamt GW_el": methan_gw,
        "Methan fest GW_el": float(leistung(n, gas, fest=True)),
        "H2 + Methan GW_el": h2_gw + methan_gw,
    })

display(pd.DataFrame(rows).round(3))

**Die H₂-Summenbedingung funktioniert: In beiden Jahren werden die vorgegebenen 10 beziehungsweise 41 GW_el exakt erreicht.**
- 2040: Methan bleibt bei 30,162 GW_el statt 50 GW_el. Die H₂-Vorgabe allein führt also nicht zur gewünschten Methanleistung.
- 2045: Der feste Methanbestand bleibt korrekt bei 19 GW_el. Einschließlich Neubau sind es 19,517 GW_el, insgesamt mit H₂ 60,517 GW_el.

## Anpassung 4 prüfen

In [0]:
rows = []

for year, gas_ziel, h2_ziel in [
    (2040, 50.0, 10.0),
    (2045, 19.0, 41.0),
]:
    n = laden(path_mod_4, year)

    gas_gw = float(leistung(n, gas))
    h2_gw = float(leistung(n, h2))

    rows.append({
        "Jahr": year,
        "Methan gesamt GW_el": gas_gw,
        "Methan Ziel GW_el": gas_ziel,
        "Methan fest GW_el": float(leistung(n, gas, fest=True)),
        "H2 gesamt GW_el": h2_gw,
        "H2 Ziel GW_el": h2_ziel,
        "Methan + H2 GW_el": gas_gw + h2_gw,
        "Ziele erreicht": (
            "ja"
            if abs(gas_gw - gas_ziel) < 0.01
            and abs(h2_gw - h2_ziel) < 0.01
            else "nein"
        ),
    })

display(pd.DataFrame(rows).round(3))

**Die Kraftwerksziele sind damit in deiner Auswertung vollständig erreicht:**
- 2040: 50 GW_el Methan + 10 GW_el H₂ = 60 GW_el.
- 2045: 19 GW_el Methan + 41 GW_el H₂ = 60 GW_el.
- Der feste Methanbestand 2045 bleibt bei 19 GW_el; zusätzlicher Methan-Neubau entfällt.

In [0]:
rows = []

for year in [2040, 2045]:
    n = laden(path_mod_2, year)
    l = n.links

    mask = (
        l.carrier.eq("H2 Electrolysis")
        & l.bus0.str.startswith("DE", na=False)
        & l.bus0.map(n.buses.carrier).eq("AC")
        & l.active
    )

    if not mask.any():
        raise ValueError(f"{year}: Keine passenden Elektrolyse-Links gefunden.")

    p = l.p_nom.where(~l.p_nom_extendable, l.p_nom_opt)
    gw = float(p.loc[mask].sum() / 1e3)

    rows.append({
        "Jahr": year,
        "Elektrolyse GW_el": gw,
        "Mindestziel GW_el": 17.0,
        "Mindestens 17 GW": "ja" if gw >= 17.0 - 0.001 else "nein",
    })

display(pd.DataFrame(rows).round(3))

### Suche CH4/ H2 Demand in Ergebnis (entspricht OG)

In [0]:
# Industrielle Verbraucher identifizieren
industrial_loads = n.loads[
    n.loads.carrier.str.contains(
        "industry|industrial",
        case=False,
        na=False
    )
]

print("Industrielle Loads:", len(industrial_loads))

display(
    industrial_loads[
        ["bus", "carrier", "p_set"]
    ].head(30)
)

# Alle zugehörigen Carrier
print(industrial_loads.carrier.value_counts())

Die Ausgabe zeigt, dass wir die relevanten Carrier gefunden haben:
- gas for industry: 27 Busse
- H2 for industry: 27 Busse

In [0]:
# Industrielle H2- und Gas-Busse auswählen
industry_buses = n.buses.index[
    n.buses.carrier.isin([
        "gas for industry",
        "H2 for industry"
    ])
]

# Loads an diesen Bussen untersuchen
industry_loads = n.loads[
    n.loads.bus.isin(industry_buses)
]

print("Anzahl industrieller Loads:", len(industry_loads))

print("\nLoad-Carrier:")
print(industry_loads.carrier.value_counts())

print("\nBeispiel-Loads:")
display(industry_loads.head(10))

Die Ausgabe zeigt:
- Carrier: gas for industry
- Komponente: Load
- Deutschland: Busse mit Präfix DE
- p_set: vorgegebene Leistung in MW
- sign = -1: Verbrauch

In [0]:
carriers = ["H2 for industry", "gas for industry"]

loads = n.loads[n.loads.carrier.isin(carriers)]

result = loads.groupby("carrier")["p_set"].sum()

result_twh = result * 8760 / 1e6

print(result_twh)

In [0]:
paths = {
    "Original": path_og,
    "Mod-1": path_mod_1,
    "Mod-2": path_mod_2,
    "Mod-3": path_mod_3,
    "Mod-4": path_mod_4,
}

years = [2025, 2030, 2035, 2040, 2045]

def h2_nachfrage(n):
    loads = n.loads[
        n.loads.carrier == "H2 for industry"
    ]
    return loads.p_set.sum() * 8760 / 1e6

def gas_nachfrage(n):
    loads = n.loads[
        n.loads.carrier == "gas for industry"
    ]
    return loads.p_set.sum() * 8760 / 1e6


ergebnisse = []

for lauf, path in paths.items():
    for year in years:
        try:
            n = laden(path, year)

            ergebnisse.append({
                "Lauf": lauf,
                "Jahr": year,
                "H2 Industrie [TWh/a]": h2_nachfrage(n),
                "Gas Industrie [TWh/a]": gas_nachfrage(n)
            })

        except FileNotFoundError:
            print(f"Nicht vorhanden: {lauf}, {year}")

df = pd.DataFrame(ergebnisse)

vergleich = df.pivot(
    index="Jahr",
    columns="Lauf",
    values=[
        "H2 Industrie [TWh/a]",
        "Gas Industrie [TWh/a]"
    ]
)

display(vergleich.reset_index().round(2))

# Test Syntax bevor Optimierung einer neuen Anpassung

In [0]:
import ast
from pathlib import Path

base = Path(
    "/Workspace/Repos/luisa.kamp@oge.net/"
    "pypsa-de-test/scripts/pypsa-de"
)

for name in ["modify_prenetwork.py", "additional_functionality.py"]:
    path = base / name
    ast.parse(path.read_text(encoding="utf-8"), filename=str(path))
    print(f"{name}: Syntax OK!")